# Knapsack instances from a simulation — QAOA vs classical solvers

This notebook uses **Mock Machines to generate optimisation test data**, then
solves it with [Qiskit Optimization](https://qiskit-community.github.io/qiskit-optimization/tutorials/09_application_classes.html#Knapsack-problem)'s
`Knapsack` application class and compares the quantum approach (QAOA on a
statevector simulator) with two classical ones.

The data comes from `examples/scenarios/KnapsackShopping` — a cut-down version of
the OnlineShopping scenario:

* **Warehouses** — one per region, seeded from `seed/Warehouse.csv`, each with a
  delivery van of `van_capacity_kg`.
* **Customers** — seeded with a *weighted* region (north 0.4, south 0.3, east 0.2,
  west 0.1). Each places exactly **one Order** for one catalog product.
* **Orders** — inherit the customer's region, sample a product (with its `price`
  and `weight_kg` from the catalog) and are routed to that region's warehouse.

At the end of a run, every warehouse holds an order book: **one 0/1 knapsack
instance per warehouse** — *which orders go on the van to maximise the value
delivered without exceeding its capacity?* Because the regions are weighted, a
single run produces instances of different sizes (the order *density* per
warehouse), and the number of customers (the seed count) scales the *volume*.

The pipeline:

```
KnapsackShopping.yaml ──mm.load / reset(N) / step──▶ Order + Warehouse tables (Arrow)
      ──▶ Knapsack(values, weights, capacity) per warehouse ──▶ QuadraticProgram
      ──▶ QAOA (quantum) │ NumPy exact eigensolver │ OR-Tools knapsack solver
```

## 1. Setup

From the root of your clone, add the package and the notebook's dependencies to
your uv project, then open this notebook:

```bash
uv add mock-machines qiskit qiskit-optimization ortools pandas matplotlib jupyter
uv run jupyter lab examples/notebooks/knapsack_qiskit.ipynb
```

In [ ]:
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import qiskit
import qiskit_optimization
from ortools.algorithms.python import knapsack_solver
from qiskit.primitives import StatevectorSampler
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_optimization.applications import Knapsack
from qiskit_optimization.converters import QuadraticProgramToQubo
from qiskit_optimization.minimum_eigensolvers import QAOA, NumPyMinimumEigensolver
from qiskit_optimization.optimizers import COBYLA
from qiskit_optimization.utils import algorithm_globals

import mockmachines as mm

SEED = 42
algorithm_globals.random_seed = SEED
print('qiskit', qiskit.__version__, '| qiskit-optimization', qiskit_optimization.__version__,
      '| mockmachines', mm.__version__)

## 2. Generate the orders with Mock Machines

`mm.load` deploys the scenario; `reset(N)` seeds a fresh run with `N` Customers (the
Warehouses always come from their seed file, one per region). Each Customer places
its Order on the first turn and the Order is routed on the next, so we step until
no Order is still `pending`.

In [ ]:
def resolve_scenario():
    env = os.environ.get('MM_SCENARIO')
    if env:
        return env
    candidates = [
        os.path.join('..', 'scenarios', 'KnapsackShopping'),
        os.path.join('examples', 'scenarios', 'KnapsackShopping'),
    ]
    return next((c for c in candidates if os.path.exists(c)), candidates[0])


SCENARIO = resolve_scenario()
sim = mm.load(SCENARIO)


def to_frame(table: pa.Table) -> pd.DataFrame:
    # Decode dictionary columns to plain strings first. Wheels up to 0.2.1 leave
    # blank entries in catalog dictionaries for unsampled products, which pandas
    # rejects as duplicate categories; once a fixed wheel ships, this can be
    # sim.observe(...).to_pandas().
    cols = [c.cast(pa.string()) if pa.types.is_dictionary(c.type) else c for c in table.columns]
    return pa.table(cols, names=table.column_names).to_pandas()


def generate(customers: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    # One simulated run: returns (orders, warehouses) as DataFrames.
    sim.reset(customers)
    for _ in range(20):
        sim.step()
        states = sim.observe('Order').column('current_state').to_pylist()
        if len(states) == customers and 'pending' not in states:
            break
    orders = to_frame(sim.observe('Order'))
    warehouses = to_frame(sim.observe('Warehouse'))
    return orders, warehouses


orders, warehouses = generate(customers=30)
orders[['ID', 'region', 'product', 'value', 'weight_kg', 'warehouse_id', 'current_state']].head(10)

In [ ]:
warehouses[['ID', 'region', 'van_capacity_kg', 'orders_received']]

The weighted regions give each warehouse a different order density — the same
run yields knapsack instances of several sizes. `orders_received` is maintained by
the simulation itself (the Warehouse's `PlaceOrder` handler), so it cross-checks
the Order → Warehouse routing:

In [ ]:
per_wh = (orders.groupby('warehouse_id')
          .agg(orders=('ID', 'size'), total_weight_kg=('weight_kg', 'sum'), total_value=('value', 'sum'))
          .join(warehouses.set_index('ID')[['region', 'van_capacity_kg', 'orders_received']]))
per_wh['load_factor'] = per_wh.total_weight_kg / per_wh.van_capacity_kg
assert (per_wh.orders == per_wh.orders_received).all()
per_wh

## 3. From order book to knapsack

For one warehouse, the items are its orders: value = product price (rounded to
whole currency units to keep the QUBO coefficients well scaled), weight =
`weight_kg`, capacity = `van_capacity_kg`. Qiskit's `Knapsack` builds a
`QuadraticProgram` with one binary variable per order and a capacity constraint.

To run on a quantum algorithm the constraint is folded into the objective as a
penalty, which needs **slack variables** — so the qubit count is the number of
orders *plus* ⌈log₂(capacity + 1)⌉. This is what bounds the instance sizes a
simulator (or today's hardware) can handle.

In [ ]:
def knapsack_for(orders: pd.DataFrame, warehouse: pd.Series) -> Knapsack:
    book = orders[orders.warehouse_id == warehouse.ID]
    return Knapsack(
        values=[int(round(v)) for v in book.value],
        weights=[int(w) for w in book.weight_kg],
        max_weight=int(warehouse.van_capacity_kg),
    )


def qubits(problem: Knapsack) -> int:
    return QuadraticProgramToQubo().convert(problem.to_quadratic_program()).get_num_vars()


# The busiest warehouse from the run above.
busiest = warehouses.loc[warehouses.orders_received.idxmax()]
kp = knapsack_for(orders, busiest)
qp = kp.to_quadratic_program()
print(f'{busiest.region}: {qp.get_num_vars()} orders -> {qubits(kp)} qubits')
print(qp.prettyprint())

## 4. Three solvers

* **QAOA** — the Quantum Approximate Optimization Algorithm, run on Qiskit's exact
  statevector sampler with a COBYLA outer loop, wrapped by `MinimumEigenOptimizer`
  (which converts the program to a QUBO). A heuristic: it returns the best
  bitstring it samples, which need not be optimal.
* **NumPy exact** — `NumPyMinimumEigensolver` diagonalises the same QUBO
  Hamiltonian classically. It is exact, and it is the reference the Qiskit
  tutorial uses, but it scales as 2^qubits.
* **OR-Tools** — Google's packaged knapsack solver (dynamic programming), the
  conventional way to solve this problem. Exact and pseudo-polynomial.

Each solver returns the value it achieved and the chosen orders; we also check
feasibility (the selection's weight against the capacity).

In [ ]:
QAOA_REPS = 1
QAOA_MAXITER = 100


def run_qaoa(problem: Knapsack):
    qaoa = QAOA(sampler=StatevectorSampler(seed=SEED), optimizer=COBYLA(maxiter=QAOA_MAXITER), reps=QAOA_REPS)
    return MinimumEigenOptimizer(qaoa).solve(problem.to_quadratic_program())


def solve_qaoa(problem: Knapsack) -> list[int]:
    return problem.interpret(run_qaoa(problem))


def solve_numpy(problem: Knapsack) -> list[int]:
    result = MinimumEigenOptimizer(NumPyMinimumEigensolver()).solve(problem.to_quadratic_program())
    return problem.interpret(result)


def solve_ortools(problem: Knapsack) -> list[int]:
    solver = knapsack_solver.KnapsackSolver(
        knapsack_solver.SolverType.KNAPSACK_DYNAMIC_PROGRAMMING_SOLVER, 'orders')
    solver.init(problem._values, [problem._weights], [problem._max_weight])
    solver.solve()
    return [i for i in range(len(problem._values)) if solver.best_solution_contains(i)]


SOLVERS = {'QAOA': solve_qaoa, 'NumPy exact': solve_numpy, 'OR-Tools': solve_ortools}


def evaluate(problem: Knapsack, chosen: list[int]) -> tuple[int, int]:
    return (sum(problem._values[i] for i in chosen), sum(problem._weights[i] for i in chosen))


chosen_by = {}
for name, solve in SOLVERS.items():
    t0 = time.perf_counter()
    chosen_by[name] = solve(kp)
    elapsed = time.perf_counter() - t0
    value, weight = evaluate(kp, chosen_by[name])
    print(f'{name:12s} value={value:4d}  weight={weight:2d}/{kp._max_weight}  '
          f'orders={chosen_by[name]}  {elapsed * 1e3:9.1f} ms')

The chosen indices map back to real simulated orders — the van manifest for the
busiest warehouse according to QAOA (QAOA is a heuristic, so on a given run it may
fall short of the OR-Tools optimum above):

In [ ]:
book = orders[orders.warehouse_id == busiest.ID].reset_index(drop=True)
book.loc[chosen_by['QAOA'], ['ID', 'product', 'value', 'weight_kg']]

## 5. Benchmark: volume and density

Now sweep the number of customers. Each run produces four instances (one per
warehouse) whose sizes follow the region weights, and repeating each volume gives
a spread of sizes and load factors. Every instance is solved by all three solvers,
except that QAOA and the NumPy eigensolver are skipped above `MAX_QUBITS` — both
simulate a 2^qubits state, and past ~16 qubits that dominates the run time.
OR-Tools solves every instance.

Two quality measures for QAOA:

* **approximation ratio** — value achieved ÷ optimal value (OR-Tools is exact), so
  1.0 means an optimal packing and an infeasible selection scores 0. This is what
  `MinimumEigenOptimizer` reports: the *best* of the ~1,000 bitstrings it samples
  from the final circuit.
* **P(optimal)** — the probability mass the final QAOA state puts on optimal
  packings. Best-of-many-samples flatters small instances (the samples cover much
  of the search space); P(optimal) shows how well the circuit itself has
  concentrated on the answer, and is the number that matters as instances outgrow
  what sampling can cover.

Note the simulation is not seeded, so each execution draws new instances.

In [ ]:
VOLUMES = [8, 16, 24, 32]   # customers per run
REPEATS = 3                  # runs per volume
MAX_QUBITS = 16

def timed(fn, *args):
    t0 = time.perf_counter()
    out = fn(*args)
    return out, time.perf_counter() - t0


instances, rows = [], []
for customers in VOLUMES:
    for rep in range(REPEATS):
        orders, warehouses = generate(customers)
        for _, wh in warehouses.iterrows():
            problem = knapsack_for(orders, wh)
            n = len(problem._values)
            instances.append(dict(customers=customers, rep=rep, region=wh.region, n_items=n))
            if n == 0:
                continue
            q = qubits(problem)
            base = dict(customers=customers, rep=rep, region=wh.region, n_items=n, qubits=q,
                        load_factor=sum(problem._weights) / problem._max_weight)

            chosen, secs = timed(solve_ortools, problem)
            optimum, _ = evaluate(problem, chosen)
            results = {'OR-Tools': (chosen, secs, np.nan)}
            if q <= MAX_QUBITS:
                results['NumPy exact'] = (*timed(solve_numpy, problem), np.nan)
                qres, secs = timed(run_qaoa, problem)
                p_opt = sum(s.probability for s in qres.samples
                            if s.status.name == 'SUCCESS' and s.fval >= optimum)
                results['QAOA'] = (problem.interpret(qres), secs, p_opt)

            for name, (chosen, secs, p_opt) in results.items():
                value, weight = evaluate(problem, chosen)
                feasible = weight <= problem._max_weight
                rows.append(dict(base, solver=name, value=value, optimum=optimum, feasible=feasible,
                                 ratio=value / optimum if feasible else 0.0, p_optimal=p_opt, seconds=secs))

instances = pd.DataFrame(instances)
bench = pd.DataFrame(rows)
print(f'{len(instances)} instances ({(instances.n_items > 0).sum()} non-empty), {len(bench)} solves')
bench.head()

**Density.** How the orders spread across warehouses as volume grows — the
region weights make north the densest and west the sparsest instance in every run:

In [ ]:
slack = int(np.ceil(np.log2(warehouses.van_capacity_kg.max() + 1)))
fig, ax = plt.subplots(figsize=(7, 3.5))
for region in ['north', 'south', 'east', 'west']:
    mean = instances[instances.region == region].groupby('customers').n_items.mean()
    ax.plot(mean.index, mean.values, marker='o', label=region)
ax.axhline(MAX_QUBITS - slack, color='grey', ls='--', lw=1)
ax.text(VOLUMES[0], MAX_QUBITS - slack + 0.3, 'largest instance run through QAOA', color='grey', fontsize=8)
ax.set(xlabel='customers per run', ylabel='orders per warehouse (mean)', title='Order density per warehouse')
ax.legend(title='region')
plt.tight_layout()

**Performance.** Run time and solution quality against instance size:

In [ ]:
fig, (ax_t, ax_r, ax_p) = plt.subplots(1, 3, figsize=(15, 4))
for name, grp in bench.groupby('solver'):
    t = grp.groupby('n_items').seconds.median()
    ax_t.plot(t.index, t.values, marker='o', label=name)
ax_t.set(yscale='log', xlabel='orders in the knapsack', ylabel='median solve time (s)', title='Run time')
ax_t.legend(loc='center right')

q = bench[bench.solver == 'QAOA']
jitter = np.random.default_rng(0).uniform(-0.15, 0.15, len(q))
ax_r.scatter(q.n_items + jitter, q.ratio, c=q.load_factor, cmap='viridis', alpha=0.8)
ax_r.set(xlabel='orders in the knapsack', ylabel='value / optimal', ylim=(-0.05, 1.05),
         title='QAOA: best sampled solution')

sc = ax_p.scatter(q.n_items + jitter, q.p_optimal, c=q.load_factor, cmap='viridis', alpha=0.8)
n = np.arange(1, q.n_items.max() + 1)
ax_p.plot(n, 0.5 ** n, color='grey', ls='--', lw=1, label='1 / 2ⁿ (one optimum, uniform guess)')
ax_p.set(yscale='log', xlabel='orders in the knapsack', ylabel='P(optimal)', title='QAOA: probability on the optimum')
ax_p.legend(fontsize=8)
fig.colorbar(sc, ax=ax_p, label='load factor (total weight / capacity)')
plt.tight_layout()

In [ ]:
summary = (bench.groupby(['solver', 'n_items'])
           .agg(instances=('ratio', 'size'), optimal=('ratio', lambda r: (r >= 1).mean()),
                p_optimal=('p_optimal', 'mean'), median_ms=('seconds', lambda s: 1e3 * s.median()))
           .round(4)
           .unstack('solver'))
summary[[('optimal', 'QAOA'), ('p_optimal', 'QAOA'),
         ('median_ms', 'QAOA'), ('median_ms', 'NumPy exact'), ('median_ms', 'OR-Tools')]]

## 6. Reading the results

* **The pipeline works end to end**: the simulation is the test-data generator.
  Changing the scenario — region weights (density), customer count (volume), the
  catalog (the weight/value mix) or van capacities (the seed file) — produces a new
  family of knapsack instances with no change to the solver code.
* **Size limits**: every order is a qubit and the capacity adds ⌈log₂(C+1)⌉ slack
  qubits, so a 15 kg van plus 12 orders is already 16 qubits. Simulated QAOA time
  grows exponentially with that count; the densest warehouses leave the quantum
  range first as volume grows.
* **Quality**: judged by its best sample, QAOA at `reps=1` usually matches the
  optimum at these sizes — but that is largely the sampling doing the work. The
  probability the circuit puts on the optimum falls exponentially as orders are
  added, roughly along the uniform-guess line: at `reps=1` the circuit barely
  concentrates on the answer, and the penalty encoding of the capacity constraint
  spreads mass over infeasible states. Once instances outgrow what ~1,000 samples
  can cover, best-of-samples will fail too. Increase `QAOA_REPS` and
  `QAOA_MAXITER` to trade run time for concentration.
* **The classical baseline**: OR-Tools' dynamic program solves every instance
  exactly in microseconds, including the ones too large to simulate. For knapsack at these sizes the
  conventional solver wins decisively; the value of the exercise is a reproducible,
  tunable benchmark harness for quantum optimisers, not a quantum advantage.

Things to try: push `MAX_QUBITS` up (or switch to `qiskit-aer`'s sampler), change
the region weights in `KnapsackShopping.yaml` for more uneven densities, or give
the warehouses different `van_capacity_kg` in `seed/Warehouse.csv`.